## Gravitino Trino Example

In this example, we will use `Jupyter` and the `Trino Python Client` to experience `Gravitino`.

In [1]:
# install trino python client and pandas
%pip install requests==2.32.3 trino==0.335.0 pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 999.1 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5
  Attempting uninstall: trino
    Found existing installation: trino 0.336.0
    Uninstalling trino-0.336.0:
      Successfully uninstalled trino-0.336.0
Note: you may need to restart the kernel to use updated packages.


In [4]:
from trino.dbapi import connect
import os

# Create a Trino connector client
conn = connect(
    host="trino",
    port=8080,
    user="admin",
    catalog="catalog_hive",
    schema="http",
)

trino_client = conn.cursor()

## Prepare

Creates a schema named `catalog_hive.company` in Hive, with its location set to`hdfs://hive:9000/user/hive/warehouse/company.db` on HDFS.

In [5]:
import os

trino_client.execute("""
CREATE SCHEMA catalog_hive.company
  WITH (location = 'hdfs://hive:9000/user/hive/warehouse/company.db')
""").fetchall()

[]

Displays the SQL command that was used to create the schema `catalog_hive.company`.

In [6]:
trino_client.execute("""
SHOW CREATE SCHEMA catalog_hive.company
""").fetchall()

[["CREATE SCHEMA catalog_hive.company\nWITH (\n   location = 'hdfs://hive:9000/user/hive/warehouse/company.db'\n)"]]

Create `employees` table

In [7]:
# Create Table
trino_client.execute(
"""
CREATE TABLE catalog_hive.company.employees
(
  name varchar,
  salary decimal(10,2)
)
WITH (
  format = 'TEXTFILE'
)
"""
).fetchall()

[]

In [8]:
# Insert data
print(trino_client.execute("INSERT INTO catalog_hive.company.employees (name, salary) VALUES ('Sam Evans', 55000)").fetchall())

[[1]]


## Simple queries

Some simple query testing.

In [9]:
import pandas as pd

# Show employees table contents
df = pd.DataFrame(trino_client.execute("SELECT * FROM catalog_hive.company.employees").fetchall(), columns=['Name', 'Salary'])

# Display the DataFrame
df

,Name,Salary
0,Sam Evans,55000.00


In [10]:
# Execute the queries and convert the results directly to DataFrames
df_g = pd.DataFrame(trino_client.execute("SHOW SCHEMAS from catalog_hive").fetchall(), columns=['Schema'])
df_g

,Schema
0,company
1,default
2,information_schema
3,product
4,sales


In [11]:
h = trino_client.execute("DESCRIBE catalog_hive.company.employees").fetchall()
h

[['name', 'varchar', '', ''], ['salary', 'decimal(10,2)', '', '']]

In [12]:
df_i = pd.DataFrame(trino_client.execute("SHOW TABLES from catalog_hive.company").fetchall(), columns=['Tables'])
df_i

,Tables
0,employees


## Cross-catalog queries

In a company, there may be different departments using different data stacks. In this example, the HR department uses Apache Hive to store its data and the sales department uses PostgreSQL. You can run some interesting queries by joining the two departments' data together with Gravitino.

To know which employee has the largest sales amount:

In [13]:
# Cross-catalog queries
cross_catalog = trino_client.execute("""
SELECT given_name, family_name, job_title, sum(total_amount) AS total_sales
FROM catalog_hive.sales.sales as s,
  catalog_postgres.hr.employees AS e
where s.employee_id = e.employee_id
GROUP BY given_name, family_name, job_title
ORDER BY total_sales DESC
LIMIT 1
""").fetchall()

# Convert the result to a DataFrame
df_j = pd.DataFrame(cross_catalog, columns=['Given Name', 'Family Name', 'Job Title', 'Total Sales'])

df_j

,Given Name,Family Name,Job Title,Total Sales
0,Dale,Lindsey,Sales Assistant,5429.61


To know the top customers who bought the most by state:

In [14]:
# Execute the query
k = trino_client.execute("""
SELECT customer_name, location, SUM(total_amount) AS total_spent
FROM catalog_hive.sales.sales AS s,
  catalog_hive.sales.stores AS l,
  catalog_hive.sales.customers AS c
WHERE s.store_id = l.store_id AND s.customer_id = c.customer_id
GROUP BY location, customer_name
ORDER BY location, SUM(total_amount) DESC
""").fetchall()

# Convert the result to a DataFrame
df_k = pd.DataFrame(k, columns=['Customer Name', 'Location', 'Total Spent'])

# Display the DataFrame
df_k

,Customer Name,Location,Total Spent
0,Harriet Best,Kansas,114201.90
1,Lenore Wilder,Kansas,43060.68
2,Raya Mcguire,Kansas,9447.84
3,Perry Tyler,Kansas,2699.73
4,Mia Hahn,Kansas,2564.19
5,Mia Hahn,Nebraska,66955.14
6,Raya Mcguire,Nebraska,45210.42
7,Erasmus Phelps,Nebraska,41952.33
8,Perry Tyler,Nebraska,37258.38
9,Harriet Best,Nebraska,28074.87


To know the employee's average performance rating and total sales:

In [15]:
# Execute the query
l = trino_client.execute("""
SELECT e.employee_id, given_name, family_name, AVG(rating) AS average_rating, SUM(total_amount) AS total_sales
FROM catalog_postgres.hr.employees AS e,
  catalog_postgres.hr.employee_performance AS p,
  catalog_hive.sales.sales AS s
WHERE e.employee_id = p.employee_id AND p.employee_id = s.employee_id
GROUP BY e.employee_id,  given_name, family_name
""").fetchall()

# Convert the result to a DataFrame
df_l = pd.DataFrame(l, columns=['Employee ID', 'Given Name', 'Family Name', 'Average Rating', 'Total Sales'])

# Display the DataFrame
df_l

,Employee ID,Given Name,Family Name,Average Rating,Total Sales
0,11,Clarke,Sanders,5.833333,9988.02
1,15,Oprah,Noel,5.000000,5879.46
2,23,Phoebe,Forbes,4.250000,5759.28
3,13,Risa,Barber,7.000000,5339.16
4,19,Xyla,Le,4.000000,815.76
5,22,Quemby,Lucas,5.000000,2692.41
6,44,Perry,Roberson,7.500000,149.94
7,49,Zephr,Long,6.000000,539.94
8,6,Jasper,Mack,4.000000,6622.92
9,25,Elijah,Burnett,3.800000,749.85


In [ ]:
import os
import signal
import ipykernel

# Lấy PID của kernel hiện tại
pid = os.getpid()

print(f"Stopping Jupyter kernel {pid} ...")
os.kill(pid, signal.SIGTERM)